# SC203 DoSSR — `test_LR` (`128x128 -> 512x512`) Baseline, SDPA On/Off Profile, Seed Study & Metrics (`01_dossr_baseline.ipynb`)

**Cell Type:** `MARKDOWN`  
**Team:** LEMMEPASS (Team 1, SC203 Scientific Methods — Class 25A02)  
**Target Branch:** `feat/baseline-runner`  
**Verification Status:** **`CELL 01`–`CELL 13` VERIFIED** (`2026-11-05`, commit `251d4a5`); **`CELL 14`–`CELL 15` (`compute_metrics.py` & export bundle) `NOT TESTED`** until Colab logs are returned.

---

## 1. Experiment Overview & Verified Discovery Summary
- **Dataset Structure (`VERIFIED` in `CELL 04`):**
  - `RealSRVal_crop128/test_LR`: **100 images at `128x128`** (low-resolution inputs).
  - `RealSRVal_crop128/test_HR`: **100 ground-truth images at `512x512`** with 1-to-1 matching filenames.
  - Earlier `2048x2048` folders (`baseline-smoke-vae-sdpa-fp16`, `seed-sanity-repeat42-fp16`, `seed-sanity-seed43-fp16`) are quarantined with `README_DO_NOT_USE_AS_BASELINE.txt`.
- **Verified `test_LR` (`128x128 -> 512x512`) Results (`CELL 08`–`CELL 13`, commit `251d4a5`):**
  1. **1-Image SDPA Patch ON vs. OFF (`CELL 08`–`CELL 09`, `VERIFIED`):**
     - **SDPA ON:** `peak_allocated = 7.32 GiB`, `peak_reserved = 8.29 GiB`, `sampler = 1.24 s`, `image = 1.75 s`.
     - **SDPA OFF (`--disable-sdpa`):** `peak_allocated = 7.32 GiB`, `peak_reserved = 8.28 GiB`, `sampler = 1.42 s`, `image = 1.92 s`.
     - **Pixel diff (SDPA ON vs. OFF, 8-bit `0..255`):** `max_abs_diff = 2.0000`, `mean_abs_diff = 0.0334`.
  2. **Output Inspection (`CELL 10`, `VERIFIED`):** `Canon_001_LR4_0.png` verified at `(512, 512)` alongside `test_HR`.
  3. **3-Image Seed Sensitivity Control (`CELL 11`, `VERIFIED`):**
     - `42 vs 42-repeat`: `max_abs_diff = 0.0000`, `mean_abs_diff = 0.0000` across all 3 images.
     - `42 vs 43`: `max_abs_diff = 255.0000`, `mean_abs_diff = 8.8237` (`5.1717`, `11.9893`, `9.3100` on `Canon_001`..`003`).
  4. **5-Image / 5-Seed Study (`CELL 12`–`CELL 13`, `VERIFIED`):** All 25 outputs (`5 images x 5 seeds: 42..46`) verified at `(512, 512)` in `seed-study-realsr-5img-lr128-fp16`.
- **New Cells (`Status: NOT TESTED`):**
  5. **Full-Reference & Cross-Seed Metrics (`CELL 14`, `NOT TESTED`):** Runs `scripts/compute_metrics.py` (`PSNR`, `SSIM`, `LPIPS` against `test_HR` plus pairwise cross-seed spread) on the 25 verified images in `seed-study-realsr-5img-lr128-fp16`.
  6. **Results Bundle Export (`CELL 15`, `NOT TESTED`):** Zips the metrics CSVs, status JSONs, comparison image, and log excerpts into `/content/sc203_baseline_results_bundle.zip` for GitHub.


---
## 2. Environment Setup & Configuration (`CELL 01`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Mounts Google Drive, sets configurable directory paths (`ASSETS_DIR`, `REPO_DIR`, `OUTPUT_BASE`), defines workflow toggles (`CACHE_CKPT_TO_LOCAL_SSD = False`, `RUN_SEED_STUDY = False`, `FORCE_SETUP = False`, `SEED_SPREAD_MIN_RATIO = 3.0`), and defines the two-pass `clean_corrupt_pngs()` and image-dimension assertion helpers.
- **Expected output:** Prints mounted paths, toggle states, and `CELL 01 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [17]:
# CELL 01 [IN-CELL RUN] — Mount Google Drive, configure paths/flags, and define helpers (Status: VERIFIED on 251d4a5)
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as exc:
    raise RuntimeError("This notebook must be executed inside Google Colab.") from exc

BRANCH = "feat/baseline-runner"
REPO_URL = "https://github.com/lemon4life/SC203.git"
REPO_DIR = Path("/content/SC203")
ASSETS_DIR = Path("/content/drive/MyDrive/lemmepass")
LOG_DIR = ASSETS_DIR / "logs"
OUTPUT_BASE = ASSETS_DIR / "outputs"
STATUS_DIR = Path("/content/sc203_status")

# Expected dimensions for RealSRVal_crop128/test_LR (128x128) at sr_scale=4 (-> 512x512)
EXPECTED_INPUT_SIZE = (128, 128)
EXPECTED_OUTPUT_SIZE = (512, 512)

# Workflow toggles
CACHE_CKPT_TO_LOCAL_SSD = False  # Default False: OS page cache already makes 2nd load fast (~1.0s)
RUN_SEED_STUDY = False           # Keep False until CELL 09-11 are reviewed and verified
FORCE_SETUP = False              # Set True to force re-running colab_setup.py
SEED_SPREAD_MIN_RATIO = 3.0      # Required ratio: mean_abs_diff(42 vs 43) >= 3.0 * mean_abs_diff(42 vs 42-repeat)

LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
STATUS_DIR.mkdir(parents=True, exist_ok=True)

CELL10_STATUS_JSON = STATUS_DIR / "cell10_inspection_status.json"
CELL11_STATUS_JSON = STATUS_DIR / "cell11_seed_sanity_status.json"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def clean_corrupt_pngs(current_output_folder: Path) -> list[str]:
    """Inspect only current_output_folder; open each PNG twice (verify + load) and delete corrupt files."""
    removed: list[str] = []
    if not current_output_folder.exists():
        return removed
    for png_path in sorted(current_output_folder.rglob("*.png")):
        if not png_path.is_file():
            continue
        try:
            with Image.open(png_path) as img_verify:
                img_verify.verify()
            with Image.open(png_path) as img_load:
                img_load.load()
        except Exception as err:
            print(f"[CORRUPT PNG REMOVED] {png_path} ({type(err).__name__}: {err})")
            png_path.unlink(missing_ok=True)
            removed.append(str(png_path))
    print(f"Checked PNGs in {current_output_folder}: removed {len(removed)} corrupt file(s).")
    return removed


def assert_folder_image_sizes(
    folder: Path,
    expected_size: tuple[int, int],
    *,
    expected_count: int | None = None,
    ignore_suffix: str = "_compare.png",
) -> list[Path]:
    """Verify every image in folder has exact expected_size (WxH); fail loudly on any mismatch."""
    if not folder.is_dir():
        raise FileNotFoundError(f"Directory not found for size check: {folder}")
    subfolders = sorted(p.name for p in folder.iterdir() if p.is_dir())
    if subfolders:
        raise ValueError(f"Directory {folder} contains unexpected subfolders: {subfolders}")
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
    files = sorted(
        p
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in exts and not p.name.endswith(ignore_suffix)
    )
    if expected_count is not None and len(files) != expected_count:
        raise RuntimeError(
            f"Expected {expected_count} image(s) in {folder}, found {len(files)}: {[p.name for p in files]}"
        )
    if not files:
        raise RuntimeError(f"No images found in {folder}")
    for p in files:
        with Image.open(p) as im:
            if im.size != expected_size:
                raise ValueError(
                    f"Image size mismatch for {p}: got {im.size}, expected {expected_size}"
                )
    print(f"Verified {len(files)} image(s) in {folder} all have size {expected_size}.")
    return files


print(f"ASSETS_DIR              : {ASSETS_DIR}")
print(f"LOG_DIR                 : {LOG_DIR}")
print(f"OUTPUT_BASE             : {OUTPUT_BASE}")
print(f"EXPECTED_INPUT_SIZE     : {EXPECTED_INPUT_SIZE}")
print(f"EXPECTED_OUTPUT_SIZE    : {EXPECTED_OUTPUT_SIZE}")
print(f"CACHE_CKPT_TO_LOCAL_SSD : {CACHE_CKPT_TO_LOCAL_SSD}")
print(f"RUN_SEED_STUDY          : {RUN_SEED_STUDY}")
print(f"SEED_SPREAD_MIN_RATIO   : {SEED_SPREAD_MIN_RATIO}")
print("CELL 01 READY")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ASSETS_DIR              : /content/drive/MyDrive/lemmepass
LOG_DIR                 : /content/drive/MyDrive/lemmepass/logs
OUTPUT_BASE             : /content/drive/MyDrive/lemmepass/outputs
EXPECTED_INPUT_SIZE     : (128, 128)
EXPECTED_OUTPUT_SIZE    : (512, 512)
CACHE_CKPT_TO_LOCAL_SSD : False
RUN_SEED_STUDY          : False
SEED_SPREAD_MIN_RATIO   : 3.0
CELL 01 READY


---
## 3. Repository Checkout & Commit Provenance (`CELL 02`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Clones or fast-forwards `/content/SC203` on `feat/baseline-runner` and dynamically prints `git rev-parse HEAD`.
- **Expected output:** Prints repository path, branch, active commit hash, and `CELL 02 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [18]:
# CELL 02 [IN-CELL RUN] — Clone or fast-forward repository and print commit hash (Status: VERIFIED on 251d4a5)
import subprocess

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

GIT_COMMIT_FULL = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
GIT_COMMIT_SHORT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"Repository : {REPO_DIR}")
print(f"Branch     : {BRANCH}")
print(f"Commit     : {GIT_COMMIT_SHORT} ({GIT_COMMIT_FULL})")
print("CELL 02 READY")


Repository : /content/SC203
Branch     : feat/baseline-runner
Commit     : 81382ae (81382ae9c3215bf67c366ab181db57f4a69824bb)
CELL 02 READY


---
## 4. Hardware, GPU, CUDA & VRAM Verification (`CELL 03`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Runs `nvidia-smi` and `free -h`, checks PyTorch CUDA and SDPA availability, prints GPU model and total/free VRAM in GiB, and warns if total GPU VRAM is below `40.0 GiB` (noting that the `2048x2048` pre-VAE-patch FP16 peak was `36.04 GiB`, `2048x2048` post-VAE-patch FP16 peak was `20.51 GiB`, and the `128x128 -> 512x512` `test_LR` peak is `NOT TESTED` yet).
- **Expected output:** GPU model name, total/free GiB, SDPA availability `True`, and `CELL 03 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 03 [IN-CELL RUN] — Verify GPU, CUDA, SDPA, and VRAM capacity (Status: VERIFIED on 251d4a5)
import platform
import subprocess
import sys
import torch
import torch.nn.functional as F

print("=== nvidia-smi ===")
subprocess.run(["nvidia-smi"], check=True)
print("\n=== System RAM (free -h) ===")
subprocess.run(["free", "-h"], check=True)

print("\n=== Python & PyTorch ===")
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SDPA available:", hasattr(F, "scaled_dot_product_attention"))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime in Colab before continuing.")
if not hasattr(F, "scaled_dot_product_attention"):
    raise RuntimeError("PyTorch scaled_dot_product_attention is required.")

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
total_gib = total_bytes / (2**30)
free_gib = free_bytes / (2**30)
print(f"GPU 0   : {props.name} | total={total_gib:.2f} GiB | free={free_gib:.2f} GiB")

VRAM_WARNING_GIB = 40.0
if total_gib < VRAM_WARNING_GIB:
    print(
        f"WARNING: GPU total VRAM ({total_gib:.2f} GiB) is below {VRAM_WARNING_GIB:.1f} GiB. "
        "On 2048x2048 outputs, FP16 peak allocated VRAM was 36.04 GiB before the VAE patch "
        "and 20.51 GiB after the VAE patch. Memory usage on 128x128 -> 512x512 test_LR is NOT TESTED yet."
    )
print("CELL 03 READY")


---
## 5. Dataset Audit (`test_LR` vs `test_HR`) & Old Output Quarantine (`CELL 04`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Verifies `RealSRVal_crop128/test_LR` (`100` images, all `128x128`) and `RealSRVal_crop128/test_HR` (`100` images, all `512x512`) and checks 1-to-1 filename pairing between `test_LR` and `test_HR`.
  2. Writes `README_DO_NOT_USE_AS_BASELINE.txt` into any earlier output directories on Drive/Colab that were generated from `test_HR` (`512x512 -> 2048x2048`) so no teammate accidentally uses them as the RealSR baseline.
- **Expected output:** Confirms 100 paired `128x128` (`test_LR`) and `512x512` (`test_HR`) images, marks any old `2048x2048` folders with `README_DO_NOT_USE_AS_BASELINE.txt`, and prints `CELL 04 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 04 [IN-CELL RUN] — Verify test_LR (128x128) & test_HR (512x512) pairing and mark old 2048x2048 outputs (Status: VERIFIED on 251d4a5)
TESTSETS_ROOT = ASSETS_DIR / "data" / "StableSR_testsets"
REALSR_ROOT = TESTSETS_ROOT / "StableSR_testsets" / "RealSRVal_crop128"
REALSR_LR_DIR = REALSR_ROOT / "test_LR"
REALSR_HR_DIR = REALSR_ROOT / "test_HR"

lr_files = assert_folder_image_sizes(REALSR_LR_DIR, EXPECTED_INPUT_SIZE, expected_count=100)
hr_files = assert_folder_image_sizes(REALSR_HR_DIR, EXPECTED_OUTPUT_SIZE, expected_count=100)

lr_names = [p.name for p in lr_files]
hr_names = [p.name for p in hr_files]
if lr_names != hr_names:
    raise RuntimeError("Filename mismatch between RealSRVal_crop128/test_LR and test_HR!")
print(f"Verified 1-to-1 filename pairing between test_LR (128x128) and test_HR (512x512) for all {len(lr_files)} images.")
print("First 3 paired filenames:", lr_names[:3])

# Mark old 2048x2048 output folders (which accidentally used test_HR 512x512 as input)
DO_NOT_USE_NOTE = (
    "DO NOT USE AS REALSR BASELINE.\n"
    "These 2048x2048 outputs were generated when input_dir pointed to the parent "
    "RealSRVal_crop128 folder, which caused rglob('*') to stage 512x512 images from "
    "test_HR instead of 128x128 images from test_LR.\n"
    "Use the *-lr128-* output folders (generated from RealSRVal_crop128/test_LR, "
    "128x128 input -> 512x512 output) for all baseline and seed-study evaluations.\n"
)
OLD_HR_INPUT_FOLDERS = [
    OUTPUT_BASE / "baseline-smoke-vae-sdpa-fp16",
    OUTPUT_BASE / "seed-sanity-repeat42-fp16",
    OUTPUT_BASE / "seed-sanity-seed43-fp16",
    OUTPUT_BASE / "baseline-realsr-5img",
    OUTPUT_BASE / "seed-study-realsr-5img",
    Path("/content/outputs/baseline-smoke-l4-fp16"),
    Path("/content/outputs/baseline-realsr-5img"),
]
for old_dir in OLD_HR_INPUT_FOLDERS:
    if old_dir.is_dir():
        marker = old_dir / "README_DO_NOT_USE_AS_BASELINE.txt"
        marker.write_text(DO_NOT_USE_NOTE, encoding="utf-8")
        print(f"Wrote {marker}")

print("CELL 04 READY")


---
## 6. Guarded Dependency Installation (`CELL 05`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Checks required module specs via `importlib.util.find_spec` and checks `external/DOSSR/scripts/inference.py`. Runs `jedi==0.20.1` + `scripts/colab_setup.py` if any module spec or `external/DOSSR` is missing or if `FORCE_SETUP = True`.
- **Expected output:** Lists checked module specs and prints `CELL 05 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [19]:
# CELL 05 [IN-CELL RUN] — Check module specs via find_spec and run guarded setup if needed (Status: VERIFIED on 251d4a5)
import importlib.util
import subprocess
import sys

REQUIRED_SPECS = (
    "pytorch_lightning",
    "open_clip",
    "omegaconf",
    "lpips",
    "einops",
    "cv2",
    "scipy",
    "transformers",
    "facexlib",
)
missing_specs = [name for name in REQUIRED_SPECS if importlib.util.find_spec(name) is None]
dossr_inference_py = REPO_DIR / "external" / "DOSSR" / "scripts" / "inference.py"

print("Missing module specs:", missing_specs)
print("DoSSR inference.py present:", dossr_inference_py.is_file())

if FORCE_SETUP or missing_specs or not dossr_inference_py.is_file():
    print("Running Jedi repair and scripts/colab_setup.py ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "jedi==0.20.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "colab_setup.py"),
            "--requirements",
            str(REPO_DIR / "external" / "DOSSR" / "requirements.txt"),
            "--log-dir",
            str(LOG_DIR),
        ],
        check=True,
    )
else:
    print("All required module specs and external/DOSSR are already present; skipping colab_setup.py.")

still_missing = [name for name in REQUIRED_SPECS if importlib.util.find_spec(name) is None]
if still_missing or not dossr_inference_py.is_file():
    raise RuntimeError(f"Setup incomplete. Missing specs: {still_missing}, DoSSR present: {dossr_inference_py.is_file()}")
print("CELL 05 READY")


Missing module specs: []
DoSSR inference.py present: True
All required module specs and external/DOSSR are already present; skipping colab_setup.py.
CELL 05 READY


---
## 7. Checkpoint Path Resolution & Optional SSD Cache (`CELL 06`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Resolves `ACTIVE_CKPT_PATH`. Default `CACHE_CKPT_TO_LOCAL_SSD = False` uses the Drive checkpoint directly. If set to `True`, copies the checkpoint to local SSD and verifies `dst.stat().st_size == src.stat().st_size` at runtime.
- **Expected output:** Prints `ACTIVE_CKPT_PATH`, byte size, and `CELL 06 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 06 [IN-CELL RUN] — Resolve checkpoint path and optional SSD cache with runtime size check (Status: VERIFIED on 251d4a5)
import shutil

DRIVE_CKPT_PATH = ASSETS_DIR / "weights" / "dossr_default.ckpt"
if not DRIVE_CKPT_PATH.is_file():
    raise FileNotFoundError(f"Missing DoSSR checkpoint on Drive: {DRIVE_CKPT_PATH}")

src_size = DRIVE_CKPT_PATH.stat().st_size
if src_size <= 0:
    raise RuntimeError(f"Drive checkpoint has invalid size ({src_size} bytes): {DRIVE_CKPT_PATH}")

if CACHE_CKPT_TO_LOCAL_SSD:
    local_ckpt = Path("/content/cache_sc203/weights/dossr_default.ckpt")
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if not local_ckpt.is_file() or local_ckpt.stat().st_size != src_size:
        print(f"Copying checkpoint to local SSD: {local_ckpt} ...")
        shutil.copyfile(DRIVE_CKPT_PATH, local_ckpt)
    dst_size = local_ckpt.stat().st_size
    if dst_size != src_size:
        local_ckpt.unlink(missing_ok=True)
        raise RuntimeError(
            f"Local checkpoint size mismatch after copy: dst={dst_size} vs src={src_size}"
        )
    ACTIVE_CKPT_PATH = local_ckpt
    print(f"Using local SSD checkpoint (verified {dst_size} bytes): {ACTIVE_CKPT_PATH}")
else:
    ACTIVE_CKPT_PATH = DRIVE_CKPT_PATH
    print(f"Using Drive checkpoint ({src_size} bytes): {ACTIVE_CKPT_PATH}")

print("CELL 06 READY")


---
## 8. GPU SDPA Numerical Equivalence Check (`CELL 07`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Runs `scripts/check_attention_sdpa.py --device cuda` on the GPU (`CrossAttention`, tiny `2x16x6x5` VAE `AttnBlock`, and larger `1x512x64x64` VAE `AttnBlock`).
- **Expected output:** Logs 5 `passed` lines with `device=cuda` and prints `CELL 07 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 07 [IN-CELL RUN] — Run CrossAttention and VAE AttnBlock SDPA check on GPU (Status: VERIFIED on 251d4a5)
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "check_attention_sdpa.py"),
        "--device",
        "cuda",
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)
print("CELL 07 READY")


---
## 9. Stage 1 `test_LR` (`128x128`) Image & Verify Input Dimensions (`CELL 08`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Stages 1 image (`Canon_001_LR4.png`) from `RealSRVal_crop128/test_LR` into `/content/realsr_smoke_1img_lr128` using `configs/baseline_realsr_lr128.yaml`.
  2. Explicitly verifies that the staged input image in `/content/realsr_smoke_1img_lr128` is **`128x128`** (failing loudly if not).
  3. Writes two configs with separate `output_root`s for the SDPA Patch ON vs. OFF comparison in `CELL 09`:
     - `/content/baseline_smoke_lr128_sdpa_on_fp16.yaml` -> `OUTPUT_BASE / "baseline-smoke-lr128-sdpa-on-fp16"`
     - `/content/baseline_smoke_lr128_sdpa_off_fp16.yaml` -> `OUTPUT_BASE / "baseline-smoke-lr128-sdpa-off-fp16"` (`disable_sdpa: true`)
- **Expected output:** Prints `Verified 1 image(s) in /content/realsr_smoke_1img_lr128 all have size (128, 128)` and `CELL 08 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 08 [IN-CELL RUN] — Stage 1 test_LR image (128x128) and prepare SDPA ON/OFF configs (Status: VERIFIED on 251d4a5)
import subprocess
import sys
import yaml

SMOKE_LR128_DIR = Path("/content/realsr_smoke_1img_lr128")
SMOKE_ON_YAML = Path("/content/baseline_smoke_lr128_sdpa_on_fp16.yaml")
SMOKE_OFF_YAML = Path("/content/baseline_smoke_lr128_sdpa_off_fp16.yaml")
SMOKE_ON_OUT_ROOT = OUTPUT_BASE / "baseline-smoke-lr128-sdpa-on-fp16"
SMOKE_OFF_OUT_ROOT = OUTPUT_BASE / "baseline-smoke-lr128-sdpa-off-fp16"

clean_corrupt_pngs(SMOKE_ON_OUT_ROOT)
clean_corrupt_pngs(SMOKE_OFF_OUT_ROOT)
SMOKE_ON_YAML.unlink(missing_ok=True)
SMOKE_OFF_YAML.unlink(missing_ok=True)

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "prepare_baseline_smoke.py"),
        "--config",
        str(REPO_DIR / "configs" / "baseline_realsr_lr128.yaml"),
        "--source-dir",
        str(REALSR_LR_DIR),
        "--image-count",
        "1",
        "--smoke-dir",
        str(SMOKE_LR128_DIR),
        "--config-out",
        str(SMOKE_ON_YAML),
        "--output-root",
        str(SMOKE_ON_OUT_ROOT),
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)

# Explicitly verify staged input is 128x128
staged_1img_files = assert_folder_image_sizes(SMOKE_LR128_DIR, EXPECTED_INPUT_SIZE, expected_count=1)

cfg_on = yaml.safe_load(SMOKE_ON_YAML.read_text(encoding="utf-8"))
cfg_on["checkpoint"] = str(ACTIVE_CKPT_PATH)
cfg_on["precision"] = "fp16"
SMOKE_ON_YAML.write_text(yaml.safe_dump(cfg_on, sort_keys=False), encoding="utf-8")

cfg_off = {
    **cfg_on,
    "output_root": str(SMOKE_OFF_OUT_ROOT),
    "disable_sdpa": True,
}
SMOKE_OFF_YAML.write_text(yaml.safe_dump(cfg_off, sort_keys=False), encoding="utf-8")

print("SDPA ON config :", SMOKE_ON_YAML, "->", SMOKE_ON_OUT_ROOT)
print("SDPA OFF config:", SMOKE_OFF_YAML, "->", SMOKE_OFF_OUT_ROOT)
print("CELL 08 READY")


---
## 10. Run 1-Image `test_LR` (`128x128 -> 512x512`) SDPA Patch ON vs. OFF Profile (`CELL 09`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Runs `Canon_001_LR4.png` (`128x128 -> 512x512`) in FP16 with `--profile-memory` and **SDPA patches ON**.
  2. Runs the same image (`128x128 -> 512x512`) in FP16 with `--profile-memory --disable-sdpa` (**SDPA patches OFF**, retaining upstream `einsum` `CrossAttention` and `bmm` `VAE AttnBlock`).
  3. Verifies both output PNGs are **`512x512`**.
  4. Prints a comparison table of `peak_allocated`, `peak_reserved`, sampler time, and per-image execution time for **SDPA ON** vs. **SDPA OFF**, and computes the pixel difference (`max_abs_diff` and `mean_abs_diff` in 8-bit `0..255` levels using `float32` / `int16`).
- **Expected output:** Comparison table of SDPA ON vs. SDPA OFF on `128x128 -> 512x512`, pixel difference between ON and OFF, verified `(512, 512)` output size, and `CELL 09 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 09 [IN-CELL RUN] — Run 1 test_LR image with SDPA ON vs SDPA OFF and compare memory/time/pixels (Status: VERIFIED on 251d4a5)
import re
import subprocess
import numpy as np
from PIL import Image

LOG_ON = Path("/content/smoke_lr128_sdpa_on_console.log")
LOG_OFF = Path("/content/smoke_lr128_sdpa_off_console.log")

clean_corrupt_pngs(SMOKE_ON_OUT_ROOT)
clean_corrupt_pngs(SMOKE_OFF_OUT_ROOT)

print("=== Running 1 test_LR image (128x128 -> 512x512) with SDPA patches ON ===")
cmd_on = (
    f"cd {REPO_DIR} && set -o pipefail && "
    f"bash scripts/run_baseline.sh --config {SMOKE_ON_YAML} --fp16 --profile-memory "
    f"2>&1 | tee {LOG_ON}"
)
ret_on = subprocess.run(["bash", "-c", cmd_on], check=False)
if ret_on.returncode != 0:
    raise RuntimeError(f"SDPA ON run failed (exit {ret_on.returncode}). Check {LOG_ON}")

print("\n=== Running 1 test_LR image (128x128 -> 512x512) with SDPA patches OFF (--disable-sdpa) ===")
cmd_off = (
    f"cd {REPO_DIR} && set -o pipefail && "
    f"bash scripts/run_baseline.sh --config {SMOKE_OFF_YAML} --fp16 --profile-memory --disable-sdpa "
    f"2>&1 | tee {LOG_OFF}"
)
ret_off = subprocess.run(["bash", "-c", cmd_off], check=False)
if ret_off.returncode != 0:
    raise RuntimeError(f"SDPA OFF run failed (exit {ret_off.returncode}). Check {LOG_OFF}")

text_on = LOG_ON.read_text(encoding="utf-8")
text_off = LOG_OFF.read_text(encoding="utf-8")

for marker in (
    "enabling CUDA float16 autocast",
    "using PyTorch scaled_dot_product_attention for vanilla CrossAttention",
    "using PyTorch scaled_dot_product_attention for vanilla VAE AttnBlock",
):
    if marker not in text_on:
        raise RuntimeError(f"SDPA ON log missing expected marker: {marker!r}")

for marker in (
    "enabling CUDA float16 autocast",
    "--disable-sdpa is set; retaining upstream einsum CrossAttention and upstream bmm VAE AttnBlock",
):
    if marker not in text_off:
        raise RuntimeError(f"SDPA OFF log missing expected marker: {marker!r}")

# Verify both output folders contain exactly 1 PNG of size 512x512
out_on_files = assert_folder_image_sizes(SMOKE_ON_OUT_ROOT / "seed_42", EXPECTED_OUTPUT_SIZE, expected_count=1)
out_off_files = assert_folder_image_sizes(SMOKE_OFF_OUT_ROOT / "seed_42", EXPECTED_OUTPUT_SIZE, expected_count=1)


def parse_profile(log_str: str) -> dict[str, str]:
    m_mem = re.search(
        r"DoSSR memory profile \| after sampler \|.*?peak_allocated=([0-9.]+) GiB \| peak_reserved=([0-9.]+) GiB",
        log_str,
    )
    m_samp = re.search(r"DoSSR sampler profile \| elapsed=([0-9.]+)s", log_str)
    m_img = re.search(r"Execution time:\s*([0-9.]+)\s*seconds", log_str)
    return {
        "peak_allocated_gib": m_mem.group(1) if m_mem else "cached/n/a",
        "peak_reserved_gib": m_mem.group(2) if m_mem else "cached/n/a",
        "sampler_s": m_samp.group(1) if m_samp else "cached/n/a",
        "image_s": m_img.group(1) if m_img else "cached/n/a",
    }


prof_on = parse_profile(text_on)
prof_off = parse_profile(text_off)

with Image.open(out_on_files[0]) as im_on, Image.open(out_off_files[0]) as im_off:
    arr_on = np.asarray(im_on.convert("RGB"), dtype=np.float32)
    arr_off = np.asarray(im_off.convert("RGB"), dtype=np.float32)
diff_on_off = np.abs(arr_on - arr_off)
max_diff_on_off = float(diff_on_off.max())
mean_diff_on_off = float(diff_on_off.mean())

print("\n=== 1-Image test_LR (128x128 -> 512x512) SDPA ON vs. OFF Comparison ===")
print(f"SDPA ON  : peak_allocated={prof_on['peak_allocated_gib']} GiB | peak_reserved={prof_on['peak_reserved_gib']} GiB | sampler={prof_on['sampler_s']} s | image={prof_on['image_s']} s")
print(f"SDPA OFF : peak_allocated={prof_off['peak_allocated_gib']} GiB | peak_reserved={prof_off['peak_reserved_gib']} GiB | sampler={prof_off['sampler_s']} s | image={prof_off['image_s']} s")
print(f"Pixel diff (SDPA ON vs SDPA OFF, 8-bit levels 0..255): max_abs_diff={max_diff_on_off:.4f} | mean_abs_diff={mean_diff_on_off:.4f}")
print("CELL 09 READY")


---
## 11. Output Inspection (`512x512`), Visual Display vs. `test_HR` & Status JSON (`CELL 10`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Runs `scripts/inspect_output.py` on the `512x512` SDPA-ON output (`--expected-size 512`).
  2. Displays the side-by-side comparison (`Bicubic 512x512` | `DoSSR 512x512`) and also displays the paired `test_HR/Canon_001_LR4.png` (`512x512` ground truth).
  3. Writes `cell10_inspection_status.json` with `passed=True`, `git_commit`, `timestamp_utc`, `checked_image_paths`, and the SDPA ON vs. OFF comparison metrics.
- **Expected output:** `OUTPUT INSPECTION PASSED`, visual comparison, and `CELL 10 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 10 [IN-CELL RUN] — Inspect 512x512 output, display alongside test_HR, write status JSON (Status: VERIFIED on 251d4a5)
import json
import subprocess
import sys
from IPython.display import display
from PIL import Image

INPUT_IMG_1 = SMOKE_LR128_DIR / "Canon_001_LR4.png"
OUTPUT_IMG_42 = SMOKE_ON_OUT_ROOT / "seed_42" / "Canon_001_LR4_0.png"
COMPARE_IMG_42 = SMOKE_ON_OUT_ROOT / "seed_42" / "Canon_001_LR4_compare.png"
GT_IMG_1 = REALSR_HR_DIR / "Canon_001_LR4.png"

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "inspect_output.py"),
        "--input",
        str(INPUT_IMG_1),
        "--output",
        str(OUTPUT_IMG_42),
        "--comparison-output",
        str(COMPARE_IMG_42),
        "--expected-size",
        str(EXPECTED_OUTPUT_SIZE[0]),
        "--seed",
        "42",
        "--overwrite",
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)

cell10_payload = {
    "passed": True,
    "git_commit": GIT_COMMIT_FULL,
    "timestamp_utc": utc_now_iso(),
    "expected_input_size": list(EXPECTED_INPUT_SIZE),
    "expected_output_size": list(EXPECTED_OUTPUT_SIZE),
    "checked_image_paths": [str(INPUT_IMG_1), str(OUTPUT_IMG_42), str(COMPARE_IMG_42)],
    "sdpa_on_vs_off": {
        "profile_on": prof_on,
        "profile_off": prof_off,
        "max_abs_diff_px": max_diff_on_off,
        "mean_abs_diff_px": mean_diff_on_off,
    },
}
CELL10_STATUS_JSON.write_text(json.dumps(cell10_payload, indent=2) + "\n", encoding="utf-8")
(OUTPUT_BASE / "cell10_inspection_status.json").write_text(
    json.dumps(cell10_payload, indent=2) + "\n", encoding="utf-8"
)

print("Displaying Bicubic (left) | DoSSR 512x512 (middle) | Ground Truth test_HR 512x512 (right):")
with Image.open(COMPARE_IMG_42) as cmp_im, Image.open(GT_IMG_1) as gt_im:
    panel = Image.new("RGB", (cmp_im.width + gt_im.width, cmp_im.height))
    panel.paste(cmp_im, (0, 0))
    panel.paste(gt_im.convert("RGB"), (cmp_im.width, 0))
    display(panel)

print(f"Wrote status file: {CELL10_STATUS_JSON}")
print("CELL 10 READY")


---
## 12. Resume Check & 3-Image Seed Sensitivity Control (`42` vs `42-repeat` vs `43`) (`CELL 11`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. **Resume Path Check:** Re-runs `scripts/run_baseline.sh` on `SMOKE_ON_OUT_ROOT` and verifies `OUTPUT CACHE | hits=1 | misses=0` with no manifest conflict.
  2. **Stage 3 `test_LR` Images (`Canon_001_LR4`..`Canon_003_LR4`):** Stages the first 3 `test_LR` images into `/content/realsr_sanity_3img_lr128` and verifies all 3 inputs have size `(128, 128)`.
  3. **Run 3 Separate Configurations (`--fp16`) into Separate `output_root`s:**
     - `seed 42` -> `OUTPUT_BASE / "seed-sanity-3img-lr128-seed42-fp16"`
     - `seed 42-repeat` (same-seed control) -> `OUTPUT_BASE / "seed-sanity-3img-lr128-repeat42-fp16"`
     - `seed 43` (cross-seed) -> `OUTPUT_BASE / "seed-sanity-3img-lr128-seed43-fp16"`
  4. **Verify Outputs & Compute Float32 Pixel Diffs:** Verifies all 3 output folders contain 3 PNGs of size `(512, 512)`. Computes per-image and overall `max_abs_diff` and `mean_abs_diff` in 8-bit pixel levels (`0..255`) using `np.float32` arrays.
  5. **Gate Rule:** If overall `mean_abs_diff(42 vs 42-repeat) == 0`, passes when overall `mean_abs_diff(42 vs 43) > 0` (and every image has `mean_abs_diff(42 vs 43) > 0`). Otherwise passes when overall `mean_abs_diff(42 vs 43) >= SEED_SPREAD_MIN_RATIO * mean_abs_diff(42 vs 42-repeat)`. Writes `cell11_seed_sanity_status.json` **first**, then raises `RuntimeError` if failed.
- **Expected output:** Per-image and 3-image aggregate diffs for `(42 vs 42-repeat)` and `(42 vs 43)`, `SEED SANITY GATE PASSED`, and `CELL 11 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 11 [IN-CELL RUN] — Test resume path and run 3-image seed control (42 vs 42-repeat vs 43) (Status: VERIFIED on 251d4a5)
import json
import subprocess
import sys
import numpy as np
import yaml
from PIL import Image

# 1. Test resume path on 1-image SMOKE_ON_OUT_ROOT
print("=== Step 1: Testing resume path on existing 1-image seed_42 output ===")
clean_corrupt_pngs(SMOKE_ON_OUT_ROOT)
resume_proc = subprocess.run(
    ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(SMOKE_ON_YAML), "--fp16"],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
    check=False,
)
print(resume_proc.stdout)
if resume_proc.returncode != 0:
    raise RuntimeError(
        f"Resume path failed (possible run_manifest.json conflict):\n{resume_proc.stdout}\n{resume_proc.stderr}"
    )
if "OUTPUT CACHE | hits=1 | misses=0" not in resume_proc.stdout:
    raise RuntimeError("Resume check did not report 'OUTPUT CACHE | hits=1 | misses=0'.")
print("Resume check passed: manifest matched and existing PNG was reused (hits=1, misses=0).")

# 2. Stage 3 test_LR images (128x128) for seed sanity check
SANITY_3IMG_DIR = Path("/content/realsr_sanity_3img_lr128")
SANITY_42_YAML = Path("/content/baseline_sanity_3img_lr128_seed42_fp16.yaml")
SANITY_42R_YAML = Path("/content/baseline_sanity_3img_lr128_repeat42_fp16.yaml")
SANITY_43_YAML = Path("/content/baseline_sanity_3img_lr128_seed43_fp16.yaml")

SANITY_42_OUT = OUTPUT_BASE / "seed-sanity-3img-lr128-seed42-fp16"
SANITY_42R_OUT = OUTPUT_BASE / "seed-sanity-3img-lr128-repeat42-fp16"
SANITY_43_OUT = OUTPUT_BASE / "seed-sanity-3img-lr128-seed43-fp16"

for out_root in (SANITY_42_OUT, SANITY_42R_OUT, SANITY_43_OUT):
    clean_corrupt_pngs(out_root)
for yml in (SANITY_42_YAML, SANITY_42R_YAML, SANITY_43_YAML):
    yml.unlink(missing_ok=True)

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "prepare_baseline_smoke.py"),
        "--config",
        str(REPO_DIR / "configs" / "baseline_realsr_lr128.yaml"),
        "--source-dir",
        str(REALSR_LR_DIR),
        "--image-count",
        "3",
        "--smoke-dir",
        str(SANITY_3IMG_DIR),
        "--config-out",
        str(SANITY_42_YAML),
        "--output-root",
        str(SANITY_42_OUT),
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)
staged_3img_files = assert_folder_image_sizes(SANITY_3IMG_DIR, EXPECTED_INPUT_SIZE, expected_count=3)

base3_cfg = yaml.safe_load(SANITY_42_YAML.read_text(encoding="utf-8"))
base3_cfg["checkpoint"] = str(ACTIVE_CKPT_PATH)
base3_cfg["precision"] = "fp16"
SANITY_42_YAML.write_text(yaml.safe_dump(base3_cfg, sort_keys=False), encoding="utf-8")

cfg_42r = {**base3_cfg, "seed": 42, "output_root": str(SANITY_42R_OUT)}
SANITY_42R_YAML.write_text(yaml.safe_dump(cfg_42r, sort_keys=False), encoding="utf-8")

cfg_43 = {**base3_cfg, "seed": 43, "output_root": str(SANITY_43_OUT)}
SANITY_43_YAML.write_text(yaml.safe_dump(cfg_43, sort_keys=False), encoding="utf-8")

for label, yml in (
    ("seed 42 (3 images)", SANITY_42_YAML),
    ("seed 42-repeat control (3 images)", SANITY_42R_YAML),
    ("seed 43 cross-seed (3 images)", SANITY_43_YAML),
):
    print(f"\n=== Running {label} ===")
    subprocess.run(
        ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(yml), "--fp16"],
        cwd=str(REPO_DIR),
        check=True,
    )

# Verify all 3 runs produced 3 outputs of size 512x512
files_42 = assert_folder_image_sizes(SANITY_42_OUT / "seed_42", EXPECTED_OUTPUT_SIZE, expected_count=3)
files_42r = assert_folder_image_sizes(SANITY_42R_OUT / "seed_42", EXPECTED_OUTPUT_SIZE, expected_count=3)
files_43 = assert_folder_image_sizes(SANITY_43_OUT / "seed_43", EXPECTED_OUTPUT_SIZE, expected_count=3)

per_image_stats = []
same_means, same_maxes = [], []
cross_means, cross_maxes = [], []

print("\n=== 3-Image Seed Sensitivity Results (8-bit pixel levels 0..255) ===")
for p42, p42r, p43 in zip(files_42, files_42r, files_43):
    if not (p42.name == p42r.name == p43.name):
        raise RuntimeError(f"Output filename mismatch: {p42.name}, {p42r.name}, {p43.name}")
    with Image.open(p42) as im42, Image.open(p42r) as im42r, Image.open(p43) as im43:
        a42 = np.asarray(im42.convert("RGB"), dtype=np.float32)
        a42r = np.asarray(im42r.convert("RGB"), dtype=np.float32)
        a43 = np.asarray(im43.convert("RGB"), dtype=np.float32)
    d_same = np.abs(a42 - a42r)
    d_cross = np.abs(a42 - a43)
    s_max, s_mean = float(d_same.max()), float(d_same.mean())
    c_max, c_mean = float(d_cross.max()), float(d_cross.mean())
    same_maxes.append(s_max)
    same_means.append(s_mean)
    cross_maxes.append(c_max)
    cross_means.append(c_mean)
    per_image_stats.append(
        {
            "image": p42.name,
            "same_seed_42_vs_42_repeat": {"max_abs_diff": s_max, "mean_abs_diff": s_mean},
            "cross_seed_42_vs_43": {"max_abs_diff": c_max, "mean_abs_diff": c_mean},
        }
    )
    print(
        f"  {p42.name}: (42 vs 42-repeat) max={s_max:.4f}, mean={s_mean:.4f} | "
        f"(42 vs 43) max={c_max:.4f}, mean={c_mean:.4f}"
    )

max_same = float(max(same_maxes))
mean_same = float(np.mean(same_means))
max_cross = float(max(cross_maxes))
mean_cross = float(np.mean(cross_means))

print(f"Aggregate (3 images): (42 vs 42-repeat) max={max_same:.4f}, mean={mean_same:.4f}")
print(f"Aggregate (3 images): (42 vs 43)        max={max_cross:.4f}, mean={mean_cross:.4f}")

if mean_same == 0.0:
    gate_passed = mean_cross > 0.0 and all(m > 0.0 for m in cross_means)
    gate_rule_desc = "mean_same == 0.0 -> require mean_cross > 0.0 for all 3 images"
else:
    gate_passed = mean_cross >= (SEED_SPREAD_MIN_RATIO * mean_same)
    gate_rule_desc = f"mean_cross >= {SEED_SPREAD_MIN_RATIO} * mean_same ({SEED_SPREAD_MIN_RATIO * mean_same:.4f})"

checked_paths = [str(p) for p in (*files_42, *files_42r, *files_43)]
cell11_payload = {
    "passed": bool(gate_passed),
    "git_commit": GIT_COMMIT_FULL,
    "timestamp_utc": utc_now_iso(),
    "seed_spread_min_ratio": SEED_SPREAD_MIN_RATIO,
    "gate_rule": gate_rule_desc,
    "checked_image_paths": checked_paths,
    "per_image": per_image_stats,
    "diff_42_vs_42_repeat_px": {"max_abs_diff": max_same, "mean_abs_diff": mean_same},
    "diff_42_vs_43_px": {"max_abs_diff": max_cross, "mean_abs_diff": mean_cross},
}
CELL11_STATUS_JSON.write_text(json.dumps(cell11_payload, indent=2) + "\n", encoding="utf-8")
(OUTPUT_BASE / "cell11_seed_sanity_status.json").write_text(
    json.dumps(cell11_payload, indent=2) + "\n", encoding="utf-8"
)
print(f"Wrote status file: {CELL11_STATUS_JSON}")

if not gate_passed:
    raise RuntimeError(
        f"SEED SANITY GATE FAILED ({gate_rule_desc}): "
        f"42 vs 42-repeat (max={max_same:.4f}, mean={mean_same:.4f}) vs "
        f"42 vs 43 (max={max_cross:.4f}, mean={mean_cross:.4f}). "
        "Stopping before multi-seed study."
    )

print("SEED SANITY GATE PASSED")
print("CELL 11 READY")


---
## 13. Stage & Run 5-Image `test_LR` (`128x128 -> 512x512`) Baseline Batch (`CELL 12`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Checks `RUN_SEED_STUDY`. Skips cleanly if `False`.
  2. Reads `cell10_inspection_status.json` and `cell11_seed_sanity_status.json` from disk, verifying `passed is True` and `git_commit == HEAD`.
  3. Stages 5 `test_LR` images (`Canon_001`..`Canon_005`) into `/content/realsr_batch_5_lr128`, verifies all 5 staged inputs are **`128x128`**, runs the batch in FP16, and verifies all 5 generated outputs are **`512x512`**.
- **Expected output:** Skips when `RUN_SEED_STUDY = False`; otherwise verifies 5 inputs (`128x128`) and 5 outputs (`512x512`) and prints `CELL 12 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 12 [IN-CELL RUN] — Disk-gated 5-image test_LR (128x128 -> 512x512) FP16 batch (Status: VERIFIED on 251d4a5)
import json
import subprocess
import sys
import yaml


def verify_disk_gates_and_commit() -> None:
    current_head = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    for label, status_path in (
        ("CELL 10 inspection", CELL10_STATUS_JSON),
        ("CELL 11 seed sanity", CELL11_STATUS_JSON),
    ):
        if not status_path.is_file():
            raise FileNotFoundError(f"Missing required status JSON on disk for {label}: {status_path}")
        data = json.loads(status_path.read_text(encoding="utf-8"))
        if data.get("passed") is not True:
            raise RuntimeError(f"{label} did not pass according to {status_path}: {data}")
        recorded_commit = data.get("git_commit")
        if recorded_commit != current_head:
            raise RuntimeError(
                f"Git commit mismatch in {status_path}: recorded {recorded_commit} != current HEAD {current_head}"
            )
        for checked_path in data.get("checked_image_paths", []):
            if not Path(checked_path).is_file():
                raise FileNotFoundError(f"Checked image recorded in {status_path} is missing: {checked_path}")
    print(f"Disk gate check passed (HEAD={current_head[:7]}).")


BATCH_5IMG_LR128_DIR = Path("/content/realsr_batch_5_lr128")
BATCH_5IMG_LR128_YAML = Path("/content/baseline_realsr_5img_lr128.yaml")
BATCH_5IMG_LR128_OUT_ROOT = OUTPUT_BASE / "baseline-realsr-5img-lr128-fp16"

if not RUN_SEED_STUDY:
    print("RUN_SEED_STUDY is False; skipping CELL 12 (5-image batch). Set RUN_SEED_STUDY = True in CELL 01 after reviewing CELL 09-11.")
else:
    verify_disk_gates_and_commit()
    clean_corrupt_pngs(BATCH_5IMG_LR128_OUT_ROOT)
    BATCH_5IMG_LR128_YAML.unlink(missing_ok=True)

    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "prepare_baseline_smoke.py"),
            "--config",
            str(REPO_DIR / "configs" / "baseline_realsr_5img_lr128.yaml"),
            "--source-dir",
            str(REALSR_LR_DIR),
            "--image-count",
            "5",
            "--smoke-dir",
            str(BATCH_5IMG_LR128_DIR),
            "--config-out",
            str(BATCH_5IMG_LR128_YAML),
            "--output-root",
            str(BATCH_5IMG_LR128_OUT_ROOT),
            "--log-dir",
            str(LOG_DIR),
        ],
        check=True,
    )

    assert_folder_image_sizes(BATCH_5IMG_LR128_DIR, EXPECTED_INPUT_SIZE, expected_count=5)

    cfg5 = yaml.safe_load(BATCH_5IMG_LR128_YAML.read_text(encoding="utf-8"))
    cfg5["checkpoint"] = str(ACTIVE_CKPT_PATH)
    cfg5["precision"] = "fp16"
    BATCH_5IMG_LR128_YAML.write_text(yaml.safe_dump(cfg5, sort_keys=False), encoding="utf-8")

    subprocess.run(
        ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(BATCH_5IMG_LR128_YAML), "--fp16"],
        cwd=str(REPO_DIR),
        check=True,
    )

    assert_folder_image_sizes(BATCH_5IMG_LR128_OUT_ROOT / "seed_42", EXPECTED_OUTPUT_SIZE, expected_count=5)
    print("CELL 12 READY")


---
## 14. Run 5-Seed Study (`seeds 42..46`) on `test_LR` (`128x128 -> 512x512`) (`CELL 13`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Checks `RUN_SEED_STUDY`. Skips cleanly if `False`.
  2. Verifies disk gate JSONs (`CELL 10` & `CELL 11`) and `git_commit == HEAD`.
  3. Verifies all 5 staged input images in `/content/realsr_batch_5_lr128` are `128x128`.
  4. Runs `scripts/run_seed_study.py --expected-input-size 128 --expected-output-size 512` across seeds `42, 43, 44, 45, 46` into `OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16"`.
  5. Verifies all 25 output PNGs (`5 seeds x 5 images`) are `512x512` and have `"precision": "fp16"` in `run_manifest.json`.
- **Expected output:** Skips when `RUN_SEED_STUDY = False`; otherwise verifies 25 `512x512` PNGs across `seed_42`..`seed_46` and prints `CELL 13 READY`.
- **Verification Status:** `VERIFIED` (`2026-11-05`, commit `251d4a5`)


In [ ]:
# CELL 13 [IN-CELL RUN] — Disk-gated 5-seed study on test_LR (128x128 -> 512x512) (Status: VERIFIED on 251d4a5)
import json
import subprocess
import sys
import yaml

SEED_STUDY_LR128_OUT_ROOT = OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16"
SEED_STUDY_LR128_YAML = Path("/content/seed_study_realsr_5img_lr128.yaml")
STUDY_SEEDS = [42, 43, 44, 45, 46]

if not RUN_SEED_STUDY:
    print("RUN_SEED_STUDY is False; skipping CELL 13 (5-seed study). Set RUN_SEED_STUDY = True in CELL 01 when ready.")
else:
    verify_disk_gates_and_commit()
    if not BATCH_5IMG_LR128_YAML.is_file():
        raise FileNotFoundError(f"Run CELL 12 first to create {BATCH_5IMG_LR128_YAML}")

    assert_folder_image_sizes(BATCH_5IMG_LR128_DIR, EXPECTED_INPUT_SIZE, expected_count=5)
    clean_corrupt_pngs(SEED_STUDY_LR128_OUT_ROOT)

    study_cfg = {
        "baseline_config": str(BATCH_5IMG_LR128_YAML),
        "seeds": STUDY_SEEDS,
        "output_root": str(SEED_STUDY_LR128_OUT_ROOT),
        "log_dir": str(LOG_DIR),
    }
    SEED_STUDY_LR128_YAML.write_text(yaml.safe_dump(study_cfg, sort_keys=False), encoding="utf-8")

    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "run_seed_study.py"),
            "--config",
            str(SEED_STUDY_LR128_YAML),
            "--baseline-config",
            str(BATCH_5IMG_LR128_YAML),
            "--expected-input-size",
            str(EXPECTED_INPUT_SIZE[0]),
            "--expected-output-size",
            str(EXPECTED_OUTPUT_SIZE[0]),
        ],
        cwd=str(REPO_DIR),
        check=True,
    )

    clean_corrupt_pngs(SEED_STUDY_LR128_OUT_ROOT)
    print("\n=== 5-Seed Study (test_LR 128x128 -> 512x512) Output Verification ===")
    for s in STUDY_SEEDS:
        s_dir = SEED_STUDY_LR128_OUT_ROOT / f"seed_{s}"
        pngs = assert_folder_image_sizes(s_dir, EXPECTED_OUTPUT_SIZE, expected_count=5)
        manifest_file = s_dir / "run_manifest.json"
        if not manifest_file.is_file():
            raise RuntimeError(f"Missing run_manifest.json in {s_dir}")
        manifest_data = json.loads(manifest_file.read_text(encoding="utf-8"))
        if manifest_data.get("precision") != "fp16":
            raise RuntimeError(f"Seed {s} manifest precision is not fp16: {manifest_data}")
        print(f"  seed_{s}: {len(pngs)} PNGs verified at {EXPECTED_OUTPUT_SIZE} | precision={manifest_data.get('precision')}")
    print("CELL 13 READY")


---
## 15. Compute Full-Reference (`PSNR`, `SSIM`, `LPIPS`) & Cross-Seed Stability Metrics (`CELL 14`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Runs `scripts/compute_metrics.py` on the 25 verified `512x512` outputs in `OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16"` (`seed_42`..`seed_46`) paired with `RealSRVal_crop128/test_HR` (`512x512`).
  2. Verifies every predicted `<stem>_0.png` and ground-truth `<stem>.png` is `(512, 512)`.
  3. Writes two CSV files into `OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16"`:
     - `metrics_per_seed.csv`: per-(image, seed) `psnr_rgb`, `ssim_rgb`, `psnr_y`, `ssim_y`, and `lpips`.
     - `metrics_summary_by_image.csv`: per-image across-seed `mean`, `std`, `min`, `max`, `range`, best/worst seed, and pairwise cross-seed pixel & LPIPS spread, plus `ALL_IMAGES_MEAN`.
  4. Note: This cell only needs `CELL 01` and `CELL 02` (plus `CELL 05` if `lpips` is not installed yet) if the 25 PNGs are already on Drive.
- **Expected output:** Prints per-image across-seed summary table, CSV paths, and `CELL 14 READY`.
- **Verification Status:** `NOT TESTED`


In [20]:
# CELL 14 [IN-CELL RUN] — Compute PSNR, SSIM, and LPIPS across seeds 42..46 against test_HR (Status: NOT TESTED)
import subprocess
import sys

REALSR_HR_DIR = ASSETS_DIR / "data" / "StableSR_testsets" / "StableSR_testsets" / "RealSRVal_crop128" / "test_HR"
SEED_STUDY_LR128_OUT_ROOT = OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16"
PER_SEED_CSV = SEED_STUDY_LR128_OUT_ROOT / "metrics_per_seed.csv"
SUMMARY_CSV = SEED_STUDY_LR128_OUT_ROOT / "metrics_summary_by_image.csv"

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "compute_metrics.py"),
        "--output-root",
        str(SEED_STUDY_LR128_OUT_ROOT),
        "--gt-dir",
        str(REALSR_HR_DIR),
        "--expected-size",
        str(EXPECTED_OUTPUT_SIZE[0]),
        "--seeds",
        "42,43,44,45,46",
        "--per-image-csv",
        str(PER_SEED_CSV),
        "--summary-csv",
        str(SUMMARY_CSV),
        "--log-dir",
        str(LOG_DIR),
    ],
    cwd=str(REPO_DIR),
    check=True,
)
print("CELL 14 READY")


CELL 14 READY


---
## 16. Package Results Bundle (`.zip`) for GitHub & Teammates (`CELL 15`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Collects lightweight, GitHub-friendly artifacts into `/content/sc203_baseline_results_bundle/`:
     - `metrics_per_seed.csv` and `metrics_summary_by_image.csv` from `CELL 14`.
     - `cell10_inspection_status.json` and `cell11_seed_sanity_status.json`.
     - `Canon_001_LR4_compare.png` (side-by-side `512x512` visual comparison).
  2. Creates `/content/sc203_baseline_results_bundle.zip` and also copies it to `OUTPUT_BASE / "sc203_baseline_results_bundle.zip"` on Google Drive.
- **Expected output:** Lists packaged files, prints zip size and Drive path, and prints `CELL 15 READY`.
- **Verification Status:** `NOT TESTED`


In [21]:
# CELL 15 [IN-CELL RUN] — Package CSVs, status JSONs, and comparison PNG into a zip bundle (Status: NOT TESTED)
import shutil
import zipfile

BUNDLE_DIR = Path("/content/sc203_baseline_results_bundle")
BUNDLE_ZIP = Path("/content/sc203_baseline_results_bundle.zip")
DRIVE_BUNDLE_ZIP = OUTPUT_BASE / "sc203_baseline_results_bundle.zip"

if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_FILES = [
    OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16" / "metrics_per_seed.csv",
    OUTPUT_BASE / "seed-study-realsr-5img-lr128-fp16" / "metrics_summary_by_image.csv",
    OUTPUT_BASE / "cell10_inspection_status.json",
    OUTPUT_BASE / "cell11_seed_sanity_status.json",
    OUTPUT_BASE / "baseline-smoke-lr128-sdpa-on-fp16" / "seed_42" / "Canon_001_LR4_compare.png",
]

copied_files = []
for src_path in CANDIDATE_FILES:
    if src_path.is_file():
        dst_path = BUNDLE_DIR / src_path.name
        shutil.copy2(src_path, dst_path)
        copied_files.append(dst_path)
    else:
        print(f"WARNING: Missing optional bundle file: {src_path}")

if not copied_files:
    raise RuntimeError("No result files found to package into bundle.")

with zipfile.ZipFile(BUNDLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in copied_files:
        zf.write(fpath, arcname=fpath.name)

shutil.copy2(BUNDLE_ZIP, DRIVE_BUNDLE_ZIP)
print(f"Packaged {len(copied_files)} file(s) into {BUNDLE_ZIP} ({BUNDLE_ZIP.stat().st_size} bytes):")
for fpath in copied_files:
    print(f"  - {fpath.name} ({fpath.stat().st_size} bytes)")
print(f"Copied bundle to Drive: {DRIVE_BUNDLE_ZIP}")
print("CELL 15 READY")


Packaged 5 file(s) into /content/sc203_baseline_results_bundle.zip (442693 bytes):
  - metrics_per_seed.csv (7210 bytes)
  - metrics_summary_by_image.csv (2159 bytes)
  - cell10_inspection_status.json (997 bytes)
  - cell11_seed_sanity_status.json (2291 bytes)
  - Canon_001_LR4_compare.png (439043 bytes)
Copied bundle to Drive: /content/drive/MyDrive/lemmepass/outputs/sc203_baseline_results_bundle.zip
CELL 15 READY


---
## 17. Troubleshooting & What to Send Back
- **Cell Type:** `MARKDOWN`
- If any cell fails, send back:
  1. The console output of the failing cell and the **last 50 lines** of the latest log in `/content/drive/MyDrive/lemmepass/logs/`.
  2. The **full Python traceback**.
